# Homelab Remake Execution Notebook
This notebook implements the homelab-remake plan by validating Docker Compose stacks, reconciling live Caddy routes, repairing homepage configuration, and bringing the homelab services up cleanly.


In [33]:
import os
import json
import subprocess
from pathlib import Path

repo_root = Path("/home/a-p-maita/homelab-config")

# Section 1: Inspect Docker environment and active stacks
print("Repo root:", repo_root)
print("Docker version:")
print(subprocess.run(["docker", "--version"], capture_output=True, text=True).stdout)
print("Docker Compose version:")
print(
    subprocess.run(
        ["docker", "compose", "version"], capture_output=True, text=True
    ).stdout
)
print("Stack directories:")
for stack_dir in sorted((repo_root / "stacks").glob("*")):
    if stack_dir.is_dir():
        print("-", stack_dir.name)


Repo root: /home/a-p-maita/homelab-config
Docker version:
Docker version 29.5.1, build 2518b52d94

Docker Compose version:
Docker Compose version 5.1.4

Stack directories:
- arr
- cloud
- downloads
- home
- infrastructure
- media
- monitoring
- services


In [ ]:
print("Backup restore guidance:")
print("Copy pre-rework ~/homelab-data into data/ instead of moving it.")
print("Example:")
print("    cd /home/a-p-maita/homelab-config")
print("    rsync -a --info=progress2 ~/homelab-data/ ./data/")
print("    sudo chown -R 1000:1000 data/")
print("Keep the original backup intact until verification is complete.")

## Section 2: Validate `.env` and Compose definitions
This section loads the repo `.env` file and validates the Compose definitions for each stack, to catch parse errors before we change anything.

In [ ]:
import dotenv
from pathlib import Path

# Load .env file from the repo root and print any variables loaded
env_path = repo_root / ".env"
print("Loading .env from:", env_path)
if not env_path.exists():
    print("ERROR: .env file not found.")
else:
    env_vars = dotenv.dotenv_values(env_path)
    print("Loaded variables:", len(env_vars))
    print("Sample keys:", sorted(list(env_vars.keys()))[:20])

# Validate each stack compose configuration
stack_paths = [
    repo_root / "stacks" / "infrastructure" / "compose.yaml",
    repo_root / "stacks" / "monitoring" / "compose.yaml",
    repo_root / "stacks" / "cloud" / "compose.db.yaml",
    repo_root / "stacks" / "cloud" / "compose.yaml",
    repo_root / "stacks" / "cloud" / "compose.override.yaml",
    repo_root / "stacks" / "services" / "compose.db.yaml",
    repo_root / "stacks" / "services" / "compose.yaml",
    repo_root / "stacks" / "media" / "compose.yaml",
    repo_root / "stacks" / "arr" / "compose.yaml",
    repo_root / "stacks" / "home" / "compose.yaml",
]
for path in stack_paths:
    if path.exists():
        print(f"Validating: {path.relative_to(repo_root)}")
        result = subprocess.run(
            [
                "docker",
                "compose",
                "--env-file",
                str(env_path),
                "-f",
                str(path),
                "config",
            ],
            capture_output=True,
            text=True,
        )
        print("Return code:", result.returncode)
        if result.returncode != 0:
            print(result.stderr)
    else:
        print(f"MISSING: {path.relative_to(repo_root)}")


## Section 3: Query running containers and health states
List active containers and identify unhealthy or exited services, focusing on caddy, authelia, homepage, and docker-socket-proxy.

In [34]:
def run(cmd):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("returncode:", result.returncode)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result


run(["docker", "ps", "--format", "table {{.Names}}\t{{.Image}}\t{{.Status}}"])
run(
    [
        "docker",
        "ps",
        "--filter",
        "health=unhealthy",
        "--format",
        "table {{.Names}}\t{{.Status}}\t{{.Image}}",
    ]
)
run(
    [
        "docker",
        "ps",
        "--filter",
        "status=exited",
        "--format",
        "table {{.Names}}\t{{.Status}}\t{{.Image}}",
    ]
)
for service in ["caddy", "authelia", "homepage", "docker-socket-proxy"]:
    print("---")
    run(
        ["docker", "inspect", "--format", "{{.Name}} {{.State.Health.Status}}", service]
    )


$ docker ps --format table {{.Names}}	{{.Image}}	{{.Status}}
returncode: 0
NAMES                 IMAGE                                                            STATUS
qbittorrent           lscr.io/linuxserver/qbittorrent:latest                           Up 2 minutes (healthy)
authelia              authelia/authelia:latest                                         Restarting (1) 10 seconds ago
forgejo               codeberg.org/forgejo/forgejo:14                                  Up 43 minutes (healthy)
homepage              ghcr.io/gethomepage/homepage:latest                              Up 43 minutes (healthy)
feishin               ghcr.io/jeffvli/feishin:latest                                   Up 54 minutes (healthy)
dockge                louislam/dockge:1                                                Up 54 minutes (healthy)
wizarr                ghcr.io/wizarrrr/wizarr:latest                                   Up 2 hours (healthy)
actual-budget         actualbudget/actual-server:lat

In [50]:
import subprocess
from pathlib import Path

with open("/tmp/homelab-diagnostics.txt", "w") as fh:
    for name in ["caddy", "homepage", "forgejo", "authelia", "docker-socket-proxy"]:
        fh.write(f"=== {name} ===\n")
        result = subprocess.run(
            [
                "docker",
                "ps",
                "--filter",
                f"name={name}",
                "--format",
                "{{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}",
            ],
            capture_output=True,
            text=True,
        )
        fh.write(result.stdout.strip() or "<no running container>")
        fh.write("\n")
        if name == "authelia":
            logs = subprocess.run(
                ["docker", "logs", "authelia", "--tail", "20"],
                capture_output=True,
                text=True,
            )
            fh.write("--- authelia logs last 20 lines ---\n")
            fh.write(logs.stdout.strip() + "\n")
        if name == "caddy":
            inspect_res = subprocess.run(
                ["docker", "inspect", "-f", "{{.NetworkSettings.Networks}}", "caddy"],
                capture_output=True,
                text=True,
            )
            fh.write("--- caddy networks ---\n")
            fh.write(inspect_res.stdout.strip() + "\n")

    fh.write("=== ENV search ===\n")
    search = subprocess.run(
        [
            "bash",
            "-lc",
            "grep -RIn 'AUTHELIA_STORAGE_ENCRYPTION_KEY\\|AUTHELIA_STORAGE_KEY\\|AUTHELIA_SESSION_SECRET' . 2>/dev/null | head -n 50",
        ],
        capture_output=True,
        text=True,
    )
    fh.write(search.stdout.strip() + "\n")

    fh.write("=== authelia config listing ===\n")
    ls = subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "-v",
            "/home/a-p-maita/homelab-config/config/authelia:/config:ro",
            "alpine",
            "sh",
            "-c",
            "ls -la /config && echo '---' && head -n 20 /config/configuration.yml",
        ],
        capture_output=True,
        text=True,
    )
    fh.write(ls.stdout.strip() + "\n")

    fh.write("=== authelia db backups search ===\n")
    find_out = subprocess.run(
        [
            "bash",
            "-lc",
            "find /home/a-p-maita/homelab-config -type f -name 'db.sqlite3*' | head -n 100",
        ],
        capture_output=True,
        text=True,
    )
    fh.write(find_out.stdout.strip() + "\n")

    fh.write("=== authelia recovery action ===\n")
    fh.write("Backing up db.sqlite3 if present...\n")
    backup = subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "-v",
            "/home/a-p-maita/homelab-config/config/authelia:/config",
            "alpine",
            "sh",
            "-c",
            "if [ -f /config/db.sqlite3 ] && [ ! -f /config/db.sqlite3.bak ]; then mv /config/db.sqlite3 /config/db.sqlite3.bak; fi; if [ -f /config/notification.txt ] && [ ! -f /config/notification.txt.bak ]; then mv /config/notification.txt /config/notification.txt.bak; fi; ls -l /config/db.sqlite3* /config/notification.txt* 2>/dev/null || true",
        ],
        capture_output=True,
        text=True,
    )
    fh.write(backup.stdout.strip() + "\n")

    fh.write("Restarting Authelia service\n")
    restart = subprocess.run(
        [
            "bash",
            "-lc",
            "cd /home/a-p-maita/homelab-config && docker compose --env-file .env -f stacks/infrastructure/compose.yaml up -d authelia",
        ],
        capture_output=True,
        text=True,
    )
    fh.write(restart.stdout.strip() + "\n")
    fh.write(restart.stderr.strip() + "\n")

print("Diagnostics written to /tmp/homelab-diagnostics.txt")


Diagnostics written to /tmp/homelab-diagnostics.txt


## Section 4: Read live Caddy autosave routes
Inspect the live Caddy autosave JSON and extract authoritative hostnames from the loaded route definitions.

In [ ]:
result = subprocess.run(
    ["docker", "exec", "caddy", "cat", "/config/caddy/autosave.json"],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print("ERROR reading Caddy autosave:", result.stderr)
else:
    data = json.loads(result.stdout)
    hosts = set()
    routes = (
        data.get("apps", {})
        .get("http", {})
        .get("servers", {})
        .get("srv", {})
        .get("routes", [])
    )
    for route in routes:
        for match in route.get("match", []):
            host = match.get("host")
            if host:
                if isinstance(host, list):
                    hosts.update(host)
                else:
                    hosts.add(host)
    print("Live Caddy hostnames:")
    for host in sorted(hosts):
        print("-", host)


## Section 5: Compare live routes to homepage YAML
Load the homepage services and settings YAML files and compare declared hostnames with the live routes from Caddy.

In [ ]:
import yaml

services_yaml = repo_root / "config" / "homepage" / "services.yaml"
runtime_services_yaml = repo_root / "data" / "homepage" / "config" / "services.yaml"

if not services_yaml.exists() or services_yaml.stat().st_size == 0:
    if runtime_services_yaml.exists():
        print(
            f"Restoring empty homepage services.yaml from runtime copy: {runtime_services_yaml}"
        )
        services_yaml.parent.mkdir(parents=True, exist_ok=True)
        services_yaml.write_text(runtime_services_yaml.read_text())
    else:
        raise FileNotFoundError(
            f"Homepage services.yaml is empty or missing and runtime copy not found: {runtime_services_yaml}"
        )

with open(services_yaml, "r") as f:
    services_data = yaml.safe_load(f) or []

print("Loaded homepage services YAML:", services_yaml)
print("Number of service groups:", len(services_data))


def collect_hosts_from_services(entries):
    hosts = set()
    for section in entries:
        if not isinstance(section, dict):
            continue
        for group_name, services in section.items():
            if isinstance(services, dict):
                for service_name, properties in services.items():
                    if isinstance(properties, dict):
                        for key in ["href", "siteMonitor"]:
                            value = properties.get(key)
                            if isinstance(value, str) and value.startswith("https://"):
                                hosts.add(value.replace("https://", ""))
    return hosts


service_hosts = collect_hosts_from_services(services_data)
print("Homepage config hosts:", len(service_hosts))
for host in sorted(service_hosts):
    print("-", host)

with open(settings_yaml, "r") as f:
    settings_data = yaml.safe_load(f)
print(
    "Loaded homepage settings layout groups:",
    sorted(settings_data.get("layout", {}).keys()),
)


In [ ]:
# Repair missing homepage service config if the canonical repo copy is empty.
canonical = repo_root / "config" / "homepage" / "services.yaml"
runtime_copy = repo_root / "data" / "homepage" / "config" / "services.yaml"
if not canonical.exists() or canonical.stat().st_size == 0:
    print(f"Restoring {canonical} from runtime copy")
    canonical.parent.mkdir(parents=True, exist_ok=True)
    canonical.write_text(runtime_copy.read_text())
    print("Restored homepage services configuration.")
else:
    print(f"{canonical} exists and is non-empty.")

## Section 6: Repair `config/homepage/services.yaml` tab ordering and duplicates
Normalize the homepage service groups and remove duplicate service entries within the same group.

In [ ]:
from collections import OrderedDict

service_order = [
    "External - Media",
    "External - Cloud & Code",
    "External - Services",
    "Streaming & Tracking",
    "Books & Reading",
    "Downloads",
    "Arr Suite",
    "Photos",
    "Notes & Documents",
    "Code & Development",
    "Finance & Food",
    "Productivity Tools",
    "Security & Access",
    "Smart Home",
    "Monitoring",
    "Infrastructure",
]

with open(services_yaml, "r") as f:
    services_data = yaml.safe_load(f)

seen = set()
new_sections = []
for section in services_data:
    if not isinstance(section, dict):
        continue
    for group_name, services in section.items():
        if group_name not in service_order:
            service_order.append(group_name)
        new_services = {}
        if isinstance(services, dict):
            for svc_name, svc_props in services.items():
                key = (
                    group_name,
                    svc_name,
                    tuple(
                        sorted(
                            (
                                svc_props.get("href") or "",
                                svc_props.get("siteMonitor") or "",
                            )
                        )
                    ),
                )
                if key in seen:
                    continue
                seen.add(key)
                new_services[svc_name] = svc_props
        new_sections.append((group_name, new_services))

# reorder groups according to the desired service_order
ordered_sections = []
for group in service_order:
    for section_group, section_services in list(new_sections):
        if section_group == group:
            ordered_sections.append({section_group: section_services})
            new_sections.remove((section_group, section_services))
for section_group, section_services in new_sections:
    ordered_sections.append({section_group: section_services})

# rewrite normalized services file with YAML block style
ordered_sections_dicts = []
for section in ordered_sections:
    if isinstance(section, dict):
        ordered_sections_dicts.append({k: dict(v) for k, v in section.items()})
    else:
        ordered_sections_dicts.append(section)

with open(services_yaml, "w") as f:
    yaml.safe_dump(ordered_sections_dicts, f, sort_keys=False)

print("Normalized services.yaml with", len(ordered_sections_dicts), "sections.")

## Section 7: Fix `config/homepage/settings.yaml` layout tab assignments
Ensure the homepage layout groups match the tab order and preserve the external/media/cloud/services/system structure.

In [ ]:
desired_layout = {
    "External - Media": {"tab": "External", "style": "row", "columns": 4},
    "External - Cloud & Code": {"tab": "External", "style": "row", "columns": 4},
    "External - Services": {"tab": "External", "style": "row", "columns": 4},
    "Streaming & Tracking": {"tab": "Media", "style": "row", "columns": 4},
    "Books & Reading": {"tab": "Media", "style": "row", "columns": 3},
    "Downloads": {"tab": "Media", "style": "row", "columns": 5},
    "Arr Suite": {"tab": "Media", "style": "row", "columns": 5},
    "Photos": {"tab": "Cloud", "style": "row", "columns": 2},
    "Notes & Documents": {"tab": "Cloud", "style": "row", "columns": 4},
    "Code & Development": {"tab": "Cloud", "style": "row", "columns": 2},
    "Finance & Food": {"tab": "Services", "style": "row", "columns": 3},
    "Productivity Tools": {"tab": "Services", "style": "row", "columns": 4},
    "Security & Access": {"tab": "Services", "style": "row", "columns": 3},
    "Smart Home": {"tab": "Services", "style": "row", "columns": 2},
    "Monitoring": {"tab": "System", "style": "row", "columns": 4},
    "Infrastructure": {"tab": "System", "style": "row", "columns": 3},
}

with open(settings_yaml, "r") as f:
    settings = yaml.safe_load(f)

settings["layout"] = desired_layout
with open(settings_yaml, "w") as f:
    yaml.safe_dump(settings, f, sort_keys=False)

print("Updated homepage settings layout to match normalized groups.")


## Section 8: Update homepage entries for live public hostnames
Adjust homepage service URLs so the external dashboard points at the authoritative live aliases reported by Caddy.

In [ ]:
live_aliases = {
    "Jellyfin": "jellyfin.andreasmaita.com",
    "Audiobookshelf": "abs.andreasmaita.com",
    "Navidrome": "music.andreasmaita.com",
    "Forgejo": "forgejo",
    "Wizarr": "join.andreasmaita.com",
    "Homepage": "homepage.andreasmaita.com",
    "Immich": "immich.andreasmaita.com",
    "Actual Budget": "actual-budget.andreasmaita.com",
    "Vaultwarden": "vault.andreasmaita.com",
    "Mealie": "mealie.andreasmaita.com",
    "Nextcloud": "nextcloud.andreasmaita.com",
    "Paperless NGX": "paperless.andreasmaita.com",
    "Joplin": "joplin.andreasmaita.com",
    "Seerr": "seerr.andreasmaita.com",
    "Auth": "auth.andreasmaita.com",
    "Yamtrack": "yamtrack.andreasmaita.com",
    "Komga": "komga.andreasmaita.com",
    "Calibre Web": "calibre-web.andreasmaita.com",
}

with open(services_yaml, "r") as f:
    services_data = yaml.safe_load(f)

changes = 0
for section in services_data:
    if not isinstance(section, dict):
        continue
    for group_name, services in section.items():
        if not isinstance(services, dict):
            continue
        for svc_name, props in services.items():
            if svc_name in live_aliases:
                host = live_aliases[svc_name]
                desired_href = f"https://{host}"
                if props.get("href") != desired_href:
                    props["href"] = desired_href
                    changes += 1
                if props.get("siteMonitor") != desired_href:
                    props["siteMonitor"] = desired_href
                    changes += 1

with open(services_yaml, "w") as f:
    yaml.safe_dump(services_data, f, sort_keys=False)

print("Updated homepage service hostnames. Total property changes:", changes)


## Section 9: Start or restart infrastructure and homepage stacks
Bring up the critical infrastructure stack and confirm the homepage service starts cleanly.

In [ ]:
for stack_cmd in [
    [
        "docker",
        "compose",
        "--env-file",
        str(repo_root / ".env"),
        "-f",
        str(repo_root / "stacks" / "infrastructure" / "compose.yaml"),
        "up",
        "-d",
    ],
    [
        "docker",
        "compose",
        "--env-file",
        str(repo_root / ".env"),
        "-f",
        str(repo_root / "stacks" / "monitoring" / "compose.yaml"),
        "up",
        "-d",
    ],
    [
        "docker",
        "compose",
        "--env-file",
        str(repo_root / ".env"),
        "-f",
        str(repo_root / "stacks" / "cloud" / "compose.db.yaml"),
        "-f",
        str(repo_root / "stacks" / "cloud" / "compose.yaml"),
        "-f",
        str(repo_root / "stacks" / "cloud" / "compose.override.yaml"),
        "up",
        "-d",
    ],
    [
        "docker",
        "compose",
        "--env-file",
        str(repo_root / ".env"),
        "-f",
        str(repo_root / "stacks" / "services" / "compose.db.yaml"),
        "up",
        "-d",
    ],
    [
        "docker",
        "compose",
        "--env-file",
        str(repo_root / ".env"),
        "-f",
        str(repo_root / "stacks" / "services" / "compose.yaml"),
        "up",
        "-d",
    ],
    [
        "docker",
        "compose",
        "--env-file",
        str(repo_root / ".env"),
        "-f",
        str(repo_root / "stacks" / "media" / "compose.yaml"),
        "up",
        "-d",
    ],
    [
        "docker",
        "compose",
        "--env-file",
        str(repo_root / ".env"),
        "-f",
        str(repo_root / "stacks" / "arr" / "compose.yaml"),
        "up",
        "-d",
    ],
    [
        "docker",
        "compose",
        "--env-file",
        str(repo_root / ".env"),
        "-f",
        str(repo_root / "stacks" / "home" / "compose.yaml"),
        "up",
        "-d",
    ],
]:
    print("Starting stack:", " ".join(stack_cmd))
    result = subprocess.run(stack_cmd, capture_output=True, text=True)
    print("Return code:", result.returncode)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)


## Section 10: Verify broken hostnames and service readiness
Test the problematic hostnames and inspect whether any onboarding or install pages are required for the services.

In [52]:
hosts_to_test = [
    "home.andreasmaita.com",
    "homepage.andreasmaita.com",
    "forgejo.andreasmaita.com",
    "forgejo",
]

for host in hosts_to_test:
    print("\n===", host, "===")
    result = subprocess.run(
        ["curl", "-I", "-H", f"Host: {host}", "http://127.0.0.1"],
        capture_output=True,
        text=True,
    )
    print("Return code:", result.returncode)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)



=== home.andreasmaita.com ===
Return code: 0
HTTP/1.1 401 Unauthorized
Content-Length: 16
Content-Type: text/plain; charset=utf-8
Date: Sun, 24 May 2026 00:57:21 GMT
Via: 1.1 Caddy


STDERR:   % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
  0     16   0      0   0      0      0      0                              0
  0     16   0      0   0      0      0      0                              0
  0     16   0      0   0      0      0      0                              0


=== homepage.andreasmaita.com ===
Return code: 0
HTTP/1.1 401 Unauthorized
Content-Length: 16
Content-Type: text/plain; charset=utf-8
Date: Sun, 24 May 2026 00:57:21 GMT
Via: 1.1 Caddy


STDERR:   % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent  

## Section 11: Clean stale homepage and Caddy config state
Confirm the home dashboard runtime config is aligned with the repo config and that the live Caddy autosave file is the authoritative route source.

In [ ]:
runtime_services_path = repo_root / "data" / "homepage" / "config" / "services.yaml"
print("Repo services.yaml:", services_yaml)
print("Runtime services.yaml:", runtime_services_path)
if runtime_services_path.exists():
    repo_checksum = subprocess.run(
        ["sha256sum", str(services_yaml)], capture_output=True, text=True
    ).stdout.split()[0]
    runtime_checksum = subprocess.run(
        ["sha256sum", str(runtime_services_path)], capture_output=True, text=True
    ).stdout.split()[0]
    print("Repo checksum:", repo_checksum)
    print("Runtime checksum:", runtime_checksum)
    if repo_checksum != runtime_checksum:
        print("checksums differ: syncing runtime homepage config from repo")
        with open(services_yaml, "r") as f:
            content = f.read()
        with open(runtime_services_path, "w") as f:
            f.write(content)
        print("Synced runtime homepage services.yaml from repo.")
    else:
        print("Runtime homepage services.yaml already matches repo.")
else:
    print("Runtime services.yaml missing; copying repo version.")
    with open(services_yaml, "r") as f:
        content = f.read()
    runtime_services_path.parent.mkdir(parents=True, exist_ok=True)
    with open(runtime_services_path, "w") as f:
        f.write(content)


## Execution note
This notebook is authored with the full homelab-remake plan, but the current notebook execution environment does not have an available Python kernel (`ipykernel` is missing), so the code cells cannot be executed here. Use the generated `scripts/homelab-remake-up.sh` script or install `ipykernel` and rerun the notebook to execute the planned recovery actions.